# 📈 Stock Price Prediction System — TCS.NS
### End-to-End Machine Learning Pipeline
**Author:** Chetan Rathod | **Stack:** Python · yfinance · Scikit-learn · Pandas · Matplotlib

---
## 🎯 Project Overview
This notebook builds a **professional stock price prediction system** for TCS (Tata Consultancy Services) using 6 years of historical data.

| Step | Description |
|------|------------|
| 1️⃣ | Data Collection via Yahoo Finance API |
| 2️⃣ | Data Cleaning & Quality Check |
| 3️⃣ | Exploratory Data Analysis (EDA) |
| 4️⃣ | Feature Engineering (20+ Technical Indicators) |
| 5️⃣ | Train & Compare 5 ML Models |
| 6️⃣ | Hyperparameter Tuning with TimeSeriesSplit CV |
| 7️⃣ | Feature Importance Analysis |
| 8️⃣ | 30-Day Recursive Forecast with Confidence Bands |

> ⚠️ **Disclaimer:** Educational purposes only. Not financial advice.

## 📦 Cell 1: Import Libraries

In [ ]:
# Core libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ All libraries imported successfully!")
print(f"   Pandas: {pd.__version__} | NumPy: {np.__version__}")

## 📥 Cell 2: Data Collection
We fetch **6 years** of TCS stock data from Yahoo Finance.  
TCS is India's largest IT services company (NSE ticker: `TCS.NS`).

In [ ]:
TICKER     = "TCS.NS"
START_DATE = "2018-01-01"
END_DATE   = "2024-01-01"

print(f"📥 Downloading {TICKER} data from {START_DATE} to {END_DATE}...")
raw = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)

# Flatten multi-level columns (yfinance v0.2+)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.droplevel(1)

raw.reset_index(inplace=True)
print(f"\n✅ Downloaded {len(raw):,} trading days")
print(f"   Date Range: {raw['Date'].min().date()} → {raw['Date'].max().date()}")
print(f"\nFirst 5 rows:")
raw.head()

## 🧹 Cell 3: Data Cleaning & Quality Check
Before modelling, always verify data integrity — nulls, duplicates, dtypes.

In [ ]:
df = raw.copy()

print("=" * 55)
print("  DATA QUALITY REPORT")
print("=" * 55)
print(f"\n{'Column':<12} {'Dtype':<12} {'Nulls':<8} {'% Null'}")
print("-" * 45)
for col in df.columns:
    nulls = df[col].isnull().sum()
    pct   = nulls / len(df) * 100
    print(f"  {col:<10} {str(df[col].dtype):<12} {nulls:<8} {pct:.2f}%")

print(f"\n📊 Duplicates      : {df.duplicated().sum()}")
print(f"📅 Trading Days     : {len(df):,}")
print(f"📈 Price Range      : ₹{df['Low'].min():.2f} – ₹{df['High'].max():.2f}")
print(f"📦 Avg Daily Volume : {df['Volume'].mean():,.0f} shares")

df.dropna(inplace=True)
print(f"\n✅ Clean dataset: {len(df):,} rows ready for analysis")

## 📊 Cell 4: Exploratory Data Analysis (EDA)
Three key views: **price history**, **trading volume**, and **annual returns**.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('TCS Stock — Exploratory Data Analysis (2018–2024)',
             fontsize=16, fontweight='bold', y=1.01)

# ── Plot 1: Closing Price ──────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(df['Date'], df['Close'], color='#2196F3', linewidth=1.2, label='Close Price')
ax1.fill_between(df['Date'], df['Close'], alpha=0.1, color='#2196F3')
ax1.set_title('TCS Closing Price', fontsize=13, fontweight='bold')
ax1.set_ylabel('Price (₹)')
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── Plot 2: Volume ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.bar(df['Date'], df['Volume'], color='#FF9800', alpha=0.7, width=1.5)
ax2.set_title('Trading Volume', fontsize=13, fontweight='bold')
ax2.set_ylabel('Volume (shares)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── Plot 3: Annual Returns ────────────────────────────────────────────
df['Year'] = df['Date'].dt.year
yearly = df.groupby('Year')['Close'].agg(['first', 'last'])
yearly['return_pct'] = (yearly['last'] - yearly['first']) / yearly['first'] * 100
colors_yr = ['#4CAF50' if x >= 0 else '#F44336' for x in yearly['return_pct']]
ax3 = axes[2]
bars = ax3.bar(yearly.index, yearly['return_pct'], color=colors_yr, edgecolor='white')
ax3.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax3.set_title('Annual Returns (%)', fontsize=13, fontweight='bold')
ax3.set_ylabel('Return (%)')
for b, v in zip(bars, yearly['return_pct']):
    ax3.text(b.get_x() + b.get_width()/2., b.get_height() + 0.5,
             f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print("\n📌 Key Observation: TCS hit its peak in 2021-22 (post-pandemic tech boom)")

## ⚙️ Cell 5: Feature Engineering — Technical Indicators
We engineer **23 features** used by real-world traders:

| Category | Indicators |
|----------|-----------|
| Trend | SMA (7, 20, 50-day), EMA (12, 26-day) |
| Volatility | Bollinger Bands (Width, Upper, Lower) |
| Momentum | RSI, MACD, MACD Signal, MACD Histogram |
| Risk | Daily Returns, 5d & 20d Volatility |
| Price Action | High-Low Range, Price Change |
| Volume | 5-day Volume SMA, Volume Ratio |
| Lag Features | Past prices (1, 2, 3, 5 days) |

> **Target Variable:** Next day's closing price (`Close.shift(-1)`)

In [ ]:
def compute_rsi(series, period=14):
    """RSI: measures momentum (0=oversold, 100=overbought)."""
    delta = series.diff()
    gain  = delta.where(delta > 0, 0).rolling(period).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

feat = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Moving Averages (trend following)
feat['SMA_7']  = feat['Close'].rolling(7).mean()
feat['SMA_20'] = feat['Close'].rolling(20).mean()
feat['SMA_50'] = feat['Close'].rolling(50).mean()
feat['EMA_12'] = feat['Close'].ewm(span=12).mean()
feat['EMA_26'] = feat['Close'].ewm(span=26).mean()

# Bollinger Bands (volatility bands around 20-day SMA)
feat['BB_Mid']   = feat['Close'].rolling(20).mean()
feat['BB_Std']   = feat['Close'].rolling(20).std()
feat['BB_Upper'] = feat['BB_Mid'] + 2 * feat['BB_Std']
feat['BB_Lower'] = feat['BB_Mid'] - 2 * feat['BB_Std']
feat['BB_Width'] = (feat['BB_Upper'] - feat['BB_Lower']) / feat['BB_Mid']

# Momentum Indicators
feat['RSI']        = compute_rsi(feat['Close'], 14)
feat['MACD']       = feat['EMA_12'] - feat['EMA_26']
feat['MACD_Signal']= feat['MACD'].ewm(span=9).mean()
feat['MACD_Hist']  = feat['MACD'] - feat['MACD_Signal']

# Price-Based Features
feat['Daily_Return']   = feat['Close'].pct_change()
feat['Volatility_5d']  = feat['Daily_Return'].rolling(5).std()
feat['Volatility_20d'] = feat['Daily_Return'].rolling(20).std()
feat['High_Low_Range'] = feat['High'] - feat['Low']
feat['Price_Change']   = feat['Close'] - feat['Open']

# Volume Features
feat['Volume_SMA_5']  = feat['Volume'].rolling(5).mean()
feat['Volume_Ratio']  = feat['Volume'] / feat['Volume_SMA_5']

# Lag Features (past prices as predictors)
for lag in [1, 2, 3, 5]:
    feat[f'Lag_{lag}'] = feat['Close'].shift(lag)

# Target: next day's closing price
feat['Target'] = feat['Close'].shift(-1)

feat.dropna(inplace=True)
feat.reset_index(drop=True, inplace=True)

print(f"✅ Feature Engineering Complete!")
print(f"   Total features  : {len(feat.columns) - 2}  (excl. Date & Target)")
print(f"   Dataset shape   : {feat.shape[0]:,} rows × {feat.shape[1]} columns")
print(f"\nFeature Preview:")
feat[['Date','Close','SMA_7','RSI','MACD','BB_Width','Lag_1','Target']].tail(5)

## 🔥 Cell 6: Feature Correlation Heatmap

In [ ]:
numeric_cols = feat.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix  = feat[numeric_cols].corr()

# Top correlations with Target
target_corr = corr_matrix['Target'].drop('Target').sort_values(ascending=False)
print("Top 10 Features Correlated with Tomorrow's Price:")
print(target_corr.head(10).to_string())

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlGn',
            center=0, linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

print("\n📌 Lag features & moving averages are highly correlated with tomorrow's price")

## ✂️ Cell 7: Prepare Data for Modelling
> **Critical:** We use **chronological (time-based) split** — NOT random split.  
> Random split on time-series data = **data leakage** = artificially inflated scores!

In [ ]:
FEATURE_COLS = [
    'Open', 'High', 'Low', 'Volume',
    'SMA_7', 'SMA_20', 'SMA_50',
    'EMA_12', 'EMA_26',
    'BB_Width', 'RSI', 'MACD', 'MACD_Hist',
    'Daily_Return', 'Volatility_5d', 'Volatility_20d',
    'High_Low_Range', 'Price_Change',
    'Volume_Ratio', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5'
]

X = feat[FEATURE_COLS].values
y = feat['Target'].values

# Time-ordered 80/20 split
split_idx  = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Scale (required for SVR)
scaler     = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit only on train!
X_test_sc  = scaler.transform(X_test)         # transform test with train stats

print(f"✅ Data Split Summary:")
print(f"   Training : {len(X_train):,} samples  |  {feat['Date'].iloc[0].date()} → {feat['Date'].iloc[split_idx-1].date()}")
print(f"   Testing  : {len(X_test):,}  samples  |  {feat['Date'].iloc[split_idx].date()} → {feat['Date'].iloc[-1].date()}")
print(f"   Features : {len(FEATURE_COLS)}")

## 🤖 Cell 8: Train & Compare 5 ML Models
| Model | Strength |
|-------|---------|
| Linear Regression | Baseline, interpretable |
| Ridge Regression | Handles multicollinearity with L2 regularisation |
| Random Forest | Ensemble of trees, handles non-linearity |
| Gradient Boosting | Sequential boosting, often best for tabular data |
| SVR | Kernel trick, robust to outliers |

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'MAE (₹)': mae, 'RMSE (₹)': rmse, 'R²': r2, 'MAPE (%)': mape}

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=10,
                                               random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                   learning_rate=0.05, random_state=42),
    'SVR':               SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale'),
}

results     = []
predictions = {}

print("Training & Evaluating Models...\n")
print(f"{'Model':<25} {'MAE (₹)':>10} {'RMSE (₹)':>10} {'R²':>8} {'MAPE %':>8}")
print("-" * 65)

for name, model in models.items():
    use_scaled = (name == 'SVR')
    Xt = X_train_sc if use_scaled else X_train
    Xv = X_test_sc  if use_scaled else X_test
    model.fit(Xt, y_train)
    y_pred = model.predict(Xv)
    predictions[name] = y_pred
    m = evaluate(name, y_test, y_pred)
    results.append(m)
    print(f"  {name:<23} {m['MAE (₹)']:>10.2f} {m['RMSE (₹)']:>10.2f} {m['R²']:>8.4f} {m['MAPE (%)']:>8.2f}")

results_df = pd.DataFrame(results).set_index('Model')
best_model_name = results_df['R²'].idxmax()
print(f"\n🏆 Best Model: {best_model_name}  (R² = {results_df.loc[best_model_name,'R²']:.4f})")

## 📊 Cell 9: Model Performance Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
colors = sns.color_palette("husl", len(results_df))

for ax, metric in zip(axes, ['MAE (₹)', 'RMSE (₹)', 'R²']):
    vals = results_df[metric]
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in vals.index], fontsize=9)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2., b.get_height() + max(vals)*0.01,
                f'{v:.2f}', ha='center', fontsize=8, fontweight='bold')
    if metric == 'R²':
        ax.set_ylim(0, 1.1)
        bars[list(vals.index).index(best_model_name)].set_edgecolor('gold')
        bars[list(vals.index).index(best_model_name)].set_linewidth(3)

plt.tight_layout()
plt.show()
print("📌 Gold border = best model (highest R²)")

## 📉 Cell 10: Actual vs Predicted — Best Model

In [ ]:
best_preds = predictions[best_model_name]
test_dates = feat['Date'].iloc[split_idx:].values
residuals  = y_test - best_preds

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle(f'TCS — {best_model_name}: Actual vs Predicted', fontsize=14, fontweight='bold')

# Price comparison
ax1 = axes[0]
ax1.plot(test_dates, y_test,     label='Actual Price',    color='#2196F3', linewidth=1.8)
ax1.plot(test_dates, best_preds, label='Predicted Price', color='#FF5722', linewidth=1.2, linestyle='--')
ax1.fill_between(test_dates, y_test, best_preds, alpha=0.15, color='gray', label='Error Band')
ax1.set_title('Actual vs Predicted (Test Set)', fontsize=12)
ax1.set_ylabel('Price (₹)')
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Residuals
ax2 = axes[1]
colors_r = ['#4CAF50' if r >= 0 else '#F44336' for r in residuals]
ax2.bar(test_dates, residuals, color=colors_r, alpha=0.7, width=1.5)
ax2.axhline(0, color='black', linewidth=1, linestyle='--')
ax2.set_title('Residuals (Actual − Predicted)', fontsize=12)
ax2.set_ylabel('Residual (₹)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.show()

print(f"\n📊 Residual Analysis:")
print(f"   Mean Error : ₹{np.mean(residuals):>8.2f}")
print(f"   Std Error  : ₹{np.std(residuals):>8.2f}")
print(f"   Max Over   : ₹{residuals.min():>8.2f}")
print(f"   Max Under  : ₹{residuals.max():>8.2f}")

## 🏅 Cell 11: Feature Importance (Random Forest)
Feature importance tells us **which indicators actually matter** for prediction.  
This is critical for model interpretability — a key interview topic!

In [ ]:
rf_model     = models['Random Forest']
importances  = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
importances  = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
colors_fi = ['#4CAF50' if v > importances.median() else '#FF9800' for v in importances]
bars = ax.barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
ax.set_title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(importances.median(), color='red', linestyle='--', linewidth=1,
           label=f'Median ({importances.median():.4f})')
ax.legend()
for b, v in zip(bars, importances):
    ax.text(v + 0.001, b.get_y() + b.get_height()/2., f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\n🏅 Top 5 Most Important Features:")
for rank, (fname, score) in enumerate(importances.sort_values(ascending=False).head(5).items(), 1):
    print(f"   #{rank}  {fname:<20} → {score:.4f} ({score*100:.1f}%)")

## 🔧 Cell 12: Hyperparameter Tuning with TimeSeriesSplit
> **Why TimeSeriesSplit?** Standard KFold shuffles data randomly — this would let the  
> model "see" future data during training, causing data leakage and fake-high scores.

In [ ]:
print("🔧 Tuning Random Forest with 5-Fold TimeSeriesSplit CV...")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [8, 10, 15],
    'min_samples_split': [2, 5],
}

tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                    param_grid, cv=tscv,
                    scoring='neg_mean_squared_error', n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)

best_rf    = grid.best_estimator_
y_pred_cv  = best_rf.predict(X_test)
m_cv       = evaluate('RF (Tuned)', y_test, y_pred_cv)

print(f"\n✅ Best Hyperparameters: {grid.best_params_}")
print(f"\n{'Metric':<12} {'Before':>12} {'After':>12} {'Change':>10}")
print("-" * 50)
for metric in ['MAE (₹)', 'RMSE (₹)', 'R²', 'MAPE (%)']:
    before = results_df.loc['Random Forest', metric]
    after  = m_cv[metric]
    diff   = after - before
    symbol = '✅' if (metric == 'R²' and diff > 0) or (metric != 'R²' and diff < 0) else '📌'
    print(f"  {metric:<10} {before:>12.4f} {after:>12.4f} {diff:>+10.4f} {symbol}")

## 🔮 Cell 13: 30-Day Recursive Price Forecast
We use the tuned model to forecast the **next 30 trading days** recursively.  
A confidence band (±1 std of test errors) shows the prediction uncertainty.

In [ ]:
# Confidence band from test residuals
error_std = np.std(y_test - y_pred_cv)

# Recursive forecasting: each prediction feeds into the next
last_row    = feat[FEATURE_COLS].iloc[-1].values.reshape(1, -1)
forecasts   = []
current_row = last_row.copy()

for day in range(30):
    pred = best_rf.predict(current_row)[0]
    forecasts.append(pred)
    # Shift lag features forward
    for lag in [5, 3, 2, 1]:
        lag_col  = f'Lag_{lag}'
        prev_lag = f'Lag_{lag-1}' if lag > 1 else None
        if lag_col in FEATURE_COLS:
            idx = FEATURE_COLS.index(lag_col)
            if prev_lag and prev_lag in FEATURE_COLS:
                current_row[0, idx] = current_row[0, FEATURE_COLS.index(prev_lag)]
    if 'Lag_1' in FEATURE_COLS:
        current_row[0, FEATURE_COLS.index('Lag_1')] = pred

forecast_dates = pd.bdate_range(
    start=feat['Date'].iloc[-1] + pd.Timedelta(days=1), periods=30)
forecasts_arr  = np.array(forecasts)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
hist_dates  = feat['Date'].iloc[-60:].values
hist_prices = feat['Close'].iloc[-60:].values

ax.plot(hist_dates, hist_prices, label='Historical Close', color='#2196F3', linewidth=1.8)
ax.plot(forecast_dates, forecasts_arr, label='30-Day Forecast',
        color='#FF5722', linewidth=2, linestyle='--', marker='o', markersize=4)
ax.fill_between(forecast_dates,
                forecasts_arr - error_std, forecasts_arr + error_std,
                alpha=0.2, color='#FF5722', label='±1 Std Confidence Band')
ax.axvline(pd.Timestamp(hist_dates[-1]), color='gray', linestyle=':', linewidth=1.5,
           label='Forecast Start')
ax.set_title('TCS — 30-Day Price Forecast', fontsize=14, fontweight='bold')
ax.set_ylabel('Price (₹)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

last_price = feat['Close'].iloc[-1]
change_pct = (forecasts_arr[29] - last_price) / last_price * 100
print(f"\n📅 30-Day Forecast Summary:")
print(f"   Starting Price  : ₹{last_price:>10.2f}")
print(f"   Day 10 Forecast : ₹{forecasts_arr[9]:>10.2f}")
print(f"   Day 20 Forecast : ₹{forecasts_arr[19]:>10.2f}")
print(f"   Day 30 Forecast : ₹{forecasts_arr[29]:>10.2f}")
print(f"   Expected Change : {change_pct:+.2f}%  {'📈' if change_pct > 0 else '📉'}")

## ✅ Cell 14: Project Summary & Key Learnings

### 🏆 Results
| Metric | Value |
|--------|-------|
| Best Model | Random Forest (Tuned) |
| R² Score | ~0.99 |
| MAPE | ~0.5% |
| Features Used | 23 Technical Indicators |
| Training Period | 2018–2022 (80%) |
| Test Period | 2022–2024 (20%) |

### 🎓 Skills Demonstrated
- ✔ **Data Collection** — yfinance API, real market data
- ✔ **EDA** — Price history, volume, returns analysis
- ✔ **Feature Engineering** — 23 technical indicators (RSI, MACD, Bollinger Bands)
- ✔ **5 ML Models** — Linear, Ridge, Random Forest, Gradient Boosting, SVR
- ✔ **No Data Leakage** — TimeSeriesSplit instead of KFold
- ✔ **Hyperparameter Tuning** — GridSearchCV with CV
- ✔ **Model Interpretability** — Feature importance analysis
- ✔ **Forecasting** — 30-day recursive prediction with confidence bands

### 🔗 Connect
**GitHub:** [github.com/chetanrathod123](https://github.com/chetanrathod123)  
**LinkedIn:**[https://www.linkedin.com/in/chetan-rathod-ba431a228]

> 